# Silver Layer - Big Data Processing with Spark

## Arquitetura Otimizada para Big Data

Este notebook implementa processamento **REAL com Spark** para volumes de dados grandes (milhões de linhas).

### Diferenças vs Versão Anterior

| Aspecto | Versão Anterior (Pandas) | Esta Versão (Spark) |
|---------|-------------------------|---------------------|
| **Leitura** | `clickhouse_connect.query_df()` | `spark.read.jdbc()` particionado |
| **Transformação** | `df.drop_duplicates()` (pandas) | `df.dropDuplicates()` (Spark) |
| **Processamento** | Em memória (single thread) | Distribuído (multi-workers) |
| **Limite** | RAM disponível (~8GB) | Praticamente ilimitado |
| **Escalabilidade** | Vertical apenas | Horizontal (adicione workers) |
| **Performance** | ~2,000 rows/s | ~50,000+ rows/s |

### Arquitetura

```

 BRONZE (ClickHouse - default schema) 
 • 3.7M+ linhas 

 
 OK Spark JDBC Read
 • Particionado (20 partitions)
 • Predicate pushdown
 • Fetchsize otimizado
 ↓

 SPARK CLUSTER (Processamento Distribuído) 
 • DataFrame API (lazy evaluation) 
 • Adaptive Query Execution (AQE) 
 • Broadcast joins para dimensões 
 • Cache inteligente 

 
 OK Spark JDBC Write
 • Bulk insert (batch 10k)
 • Particionado
 ↓

 SILVER (ClickHouse - trusted) 
 • Dados limpos e validados 
 • MergeTree engine 

```

### Performance Esperada

- **100k rows**: ~2s
- **1M rows**: ~15s
- **10M rows**: ~2.5min
- **100M rows**: ~25min

---
## 1. Imports e Configuração

In [243]:
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any
import json

from pyspark.sql import SparkSession, DataFrame as SparkDataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import clickhouse_connect
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

load_dotenv()
print("Imports carregados")

Imports carregados


---
## 2. Logging e Observabilidade

In [244]:
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s:%(funcName)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Setup directories
current_dir = Path(os.getcwd()) if 'os' in dir() else Path('.')
if current_dir.name == "jupyter-notebook":
    BASE_DIR = current_dir.parent.parent
elif current_dir.name == "output":
    BASE_DIR = current_dir.parent
else:
    BASE_DIR = current_dir

LOGS_DIR = BASE_DIR / "logs" / "silver"
METRICS_DIR = BASE_DIR / "output" / "metrics" / "silver"

LOGS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# File handler for structured logging
from datetime import datetime
file_handler = logging.FileHandler(
    LOGS_DIR / f"silver_pipeline_{datetime.now():%Y%m%d_%H%M%S}.log"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter(
    '%(asctime)s | %(levelname)s | %(name)s:%(funcName)s | %(message)s'
))
logger.addHandler(file_handler)

logger.info("Logging system configured")
logger.info(f"Logs directory: {LOGS_DIR}")
logger.info(f"Metrics directory: {METRICS_DIR}")

print("Logging configured successfully")

2026-02-10 20:28:15 | INFO     | __main__:<module> | Logging system configured
2026-02-10 20:28:15 | INFO     | __main__:<module> | Logs directory: /app/logs/silver
2026-02-10 20:28:15 | INFO     | __main__:<module> | Metrics directory: /app/output/metrics/silver


Logging configured successfully


---
## 1.6. Metrics Collector Class

In [245]:
from dataclasses import dataclass, asdict
from typing import Optional
from datetime import datetime


@dataclass
class PipelineMetrics:
    """Métricas do pipeline de processamento"""
    table_name: str
    start_time: datetime
    end_time: Optional[datetime] = None
 
    # Volumetria
    rows_input: int = 0
    rows_output: int = 0
    rows_duplicates: int = 0
    rows_invalid: int = 0
    rows_nulls: int = 0
 
    # Performance
    duration_seconds: float = 0.0
    throughput_rows_per_sec: float = 0.0
 
    # Qualidade
    quality_score: float = 0.0
    quality_checks_passed: int = 0
    quality_checks_failed: int = 0
 
    # Status
    status: str = "running"
    error_message: Optional[str] = None
 
    def finalize(self):
     """Finaliza as métricas calculando valores derivados"""
     self.end_time = datetime.now()
     self.duration_seconds = (self.end_time - self.start_time).total_seconds()
     
     if self.duration_seconds > 0:
         self.throughput_rows_per_sec = self.rows_output / self.duration_seconds
     
     total_checks = self.quality_checks_passed + self.quality_checks_failed
     if total_checks > 0:
         self.quality_score = (self.quality_checks_passed / total_checks) * 100
 
    def to_dict(self) -> Dict:
     """Converte para dicionário serializável"""
     data = asdict(self)
     data['start_time'] = self.start_time.isoformat()
     data['end_time'] = self.end_time.isoformat() if self.end_time else None
     return data


class MetricsCollector:
    """Coletor centralizado de métricas"""
 
    def __init__(self, output_dir: Path):
     self.output_dir = output_dir
     self.metrics: List[PipelineMetrics] = []
     logger.info(f"MetricsCollector inicializado: {output_dir}")
 
    def add_metric(self, metric: PipelineMetrics):
     """Adiciona métrica à coleção"""
     self.metrics.append(metric)
     logger.debug(f"Métrica adicionada: {metric.table_name}")
 
    def save_metrics(self, filename: str = None):
     """Salva métricas em JSON"""
     if not filename:
         filename = f"pipeline_metrics_{datetime.now():%Y%m%d_%H%M%S}.json"
     filepath = self.output_dir / filename
     data = {
         'pipeline_run': datetime.now().isoformat(),
         'total_tables': len(self.metrics),
         'metrics': [m.to_dict() for m in self.metrics]
     }
     with open(filepath, 'w') as f:
         json.dump(data, f, indent=2)
     logger.info(f"Métricas salvas: {filepath}")
     return filepath
 
    def get_summary(self) -> Dict:
     """Retorna sumário das métricas"""
     if not self.metrics:
         return {}
     total_rows_input = sum(m.rows_input for m in self.metrics)
     total_rows_output = sum(m.rows_output for m in self.metrics)
     total_duplicates = sum(m.rows_duplicates for m in self.metrics)
     total_invalid = sum(m.rows_invalid for m in self.metrics)
     avg_quality = np.mean([m.quality_score for m in self.metrics if m.quality_score > 0])
     return {
         'total_tables_processed': len(self.metrics),
         'total_rows_input': total_rows_input,
         'total_rows_output': total_rows_output,
         'total_duplicates_removed': total_duplicates,
         'total_invalid_rows': total_invalid,
         'avg_quality_score': round(avg_quality, 2),
         'success_rate': round((len([m for m in self.metrics if m.status == 'success']) / len(self.metrics)) * 100, 2)
     }


# Inicializar coletor
metrics_collector = MetricsCollector(METRICS_DIR)
logger.info("MetricsCollector inicializado")

2026-02-10 20:28:15 | INFO     | __main__:__init__ | MetricsCollector inicializado: /app/output/metrics/silver
2026-02-10 20:28:15 | INFO     | __main__:<module> | MetricsCollector inicializado


---
## 2. Inicializar Spark com Configurações Big Data

In [246]:
CH_HOST = os.getenv('CLICKHOUSE_HOST', 'e1a1lieug8.us-central1.gcp.clickhouse.cloud')
CH_PORT = int(os.getenv('CLICKHOUSE_PORT', 8443))
CH_USER = os.getenv('CLICKHOUSE_USER', 'default')
CH_PASSWORD = os.getenv('CLICKHOUSE_PASSWORD', '_uv765EvWphL_')

CH_DATABASE_BRONZE = 'raw'
CH_DATABASE_SILVER = 'trusted'

JDBC_URL = f"jdbc:clickhouse:https://{CH_HOST}:{CH_PORT}/{CH_DATABASE_BRONZE}?ssl=true"

print(f" ClickHouse JDBC: {JDBC_URL}")
print(f"Bronze: {CH_DATABASE_BRONZE}")
print(f"Silver: {CH_DATABASE_SILVER}")

 ClickHouse JDBC: jdbc:clickhouse:https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443/raw?ssl=true
Bronze: raw
Silver: trusted


In [247]:
print("Spark Inicializando Spark com configurações Big Data...\n")

spark = SparkSession.builder \
    .appName("SilverLayer-BigData-Optimized") \
    .config("spark.jars.packages", "com.clickhouse:clickhouse-jdbc:0.4.6,com.clickhouse:clickhouse-client:0.4.6") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.adaptive.localShuffleReader.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "100") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.fallback.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Success: Spark inicializado!")
print(f" Versão: {spark.version}")
print(f" Parallelism: {spark.sparkContext.defaultParallelism}")
print(f" Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f" AQE Habilitado: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f" Arrow Habilitado: {spark.conf.get('spark.sql.execution.arrow.pyspark.enabled')}")

Spark Inicializando Spark com configurações Big Data...

Success: Spark inicializado!
 Versão: 3.4.1
 Parallelism: 100
 Shuffle Partitions: 200
 AQE Habilitado: true
 Arrow Habilitado: true


---
## 3. Conexão ClickHouse (Metadata)

In [248]:
# Cliente ClickHouse apenas para metadata e métricas
print(f" Conectando ao ClickHouse: {CH_HOST}:{CH_PORT}")

client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
    secure=True
)

version = client.query("SELECT version()").result_rows[0][0]
print(f"Success: ClickHouse {version}")

# Criar database Silver
client.command(f"CREATE DATABASE IF NOT EXISTS {CH_DATABASE_SILVER}")
print(f" Database Silver: {CH_DATABASE_SILVER} pronta")

# Criar tabelas de métricas
client.command(f"""
    CREATE TABLE IF NOT EXISTS {CH_DATABASE_SILVER}.spark_processing_metrics (
    execution_id String,
    table_name String,
    execution_timestamp DateTime,
    start_time DateTime,
    end_time DateTime,
    duration_seconds Float32,
    rows_input UInt64,
    rows_output UInt64,
    rows_duplicates UInt64,
    num_partitions UInt32,
    throughput_rows_per_sec Float32,
    status String,
    error_message String
    ) ENGINE = MergeTree()
    ORDER BY (table_name, execution_timestamp)
""")

print("Success: Tabelas de métricas criadas")

 Conectando ao ClickHouse: e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443
Success: ClickHouse 25.12.1.1370
 Database Silver: trusted pronta
Success: Tabelas de métricas criadas


---
## 4. Definir Tabelas para Processar

In [249]:
# Listar tabelas Bronze
tables_info = []

tables_to_process = [
    "ginf.depara_cliente",
    "ginf.BASE_CEP_COMPLETA",
    "bistage.TST_CONTRATOS_BI",
    "ginf.BASE_REGIONAL",
    "ginf.TAB_CIDADE_DELITO_SP_CAP",
    "siga.SC5030",
    "siga.SC6030",
    "ginf.TST_HISTORICO_SOLICITACOES",
    "ginf.TST_SOLICIT_CADASTRADAS",
    "siga.SD2030",
    "siga.CN9030",
    "siga.SA1030",
    "siga.SA3030",
    "siga.SB1030",
    "siga.SZH030",
    "siga.SZJ030",
    "siga.SZU030",
    "siga.SZV030",
    "siga.SZW030",
    "siga.ZAA030",
    "siga.ZA1030",
    "siga.ZA3030",
    "siga.ZB3030",
    "siga.ZE8030",
    "siga.ZTX030",
    "siga.ZT1030",
    "siga.CN1030",
    "siga.CNB030",
    "siga.SE4030",
    "siga.SF2030",
    "ginf.TST_CONTRATOS",
    "scot.ERP_PRODUCT",
    "scot.ERP_PRODUCT_ITEM",
    "scot.ERP_VEHICLE",
    "scot.ERP_AGREEMENT",
    "scot.SC_CITY",
    "scot.SC_GROUP",
    "scot.SC_LOCATION",
    "scot.SC_REQ_FILE",
    "scot.SC_REQUISITION",
    "scot.SC_REQUISITION_HISTORY",
    "scot.SC_REQUISITION_QUEUE",
    "scot.SC_REQUISITION_STATUS",
    "scot.SC_RESERVE",
    "scot.SC_RESERVE_LOCATION",
    "scot.SC_RESULT_CODE",
    "scot.SC_ROLE",
    "scot.SC_STATE",
    "scot.CEPREG",
    "scot.SC_TASK",
    "scot.SC_WAREHOUSE",
    "scot.SC_TECHNICAL_REGISTER",
    "scot.SC_WEBSERVICE_REQUISITION",
]

print(" Verificando tabelas Bronze...\n")

for table in tables_to_process:
     try:
         bronze_table = table.split(".", 1)[-1].lower()
         result = client.query(f"SELECT count() FROM {CH_DATABASE_BRONZE}.{bronze_table}")
         count = result.result_rows[0][0]
         if count > 0:
             tables_info.append({
                 'table_name': table,
                 'total_rows': count,
                 'status': 'OK'
             })
             print(f"Success: {table}: {count:,} linhas")
     except Exception as e:
         print(f"Error {table}: {str(e)[:200]}")

df_tables = pd.DataFrame(tables_info)
total_rows = df_tables['total_rows'].sum() if 'total_rows' in df_tables.columns else 0
print(f"\n Total: {len(tables_info)} tabelas, {total_rows:,} linhas")
df_tables

 Verificando tabelas Bronze...

Success: ginf.depara_cliente: 47 linhas
Success: ginf.BASE_CEP_COMPLETA: 96,778 linhas
Success: bistage.TST_CONTRATOS_BI: 100,000 linhas
Success: ginf.BASE_REGIONAL: 27 linhas
Success: ginf.TAB_CIDADE_DELITO_SP_CAP: 45 linhas
Success: siga.SC5030: 50,000 linhas
Success: siga.SC6030: 50,000 linhas
Success: ginf.TST_HISTORICO_SOLICITACOES: 100,000 linhas
Success: ginf.TST_SOLICIT_CADASTRADAS: 50,130 linhas
Success: siga.SD2030: 50,000 linhas
Success: siga.CN9030: 100,000 linhas
Success: siga.SA1030: 100,000 linhas
Success: siga.SA3030: 3,104 linhas
Success: siga.SB1030: 9,316 linhas
Success: siga.SZH030: 17 linhas
Success: siga.SZJ030: 73 linhas
Success: siga.SZU030: 100,000 linhas
Success: siga.SZV030: 100,000 linhas
Success: siga.SZW030: 16 linhas
Success: siga.ZAA030: 50,000 linhas
Success: siga.ZA1030: 50,000 linhas
Success: siga.ZA3030: 50,000 linhas
Success: siga.ZB3030: 62 linhas
Success: siga.ZE8030: 35 linhas
Success: siga.ZTX030: 100,000 linhas
S

,table_name,total_rows,status
0,ginf.depara_cliente,47,OK
1,ginf.BASE_CEP_COMPLETA,96778,OK
2,bistage.TST_CONTRATOS_BI,100000,OK
3,ginf.BASE_REGIONAL,27,OK
4,ginf.TAB_CIDADE_DELITO_SP_CAP,45,OK
5,siga.SC5030,50000,OK
6,siga.SC6030,50000,OK
7,ginf.TST_HISTORICO_SOLICITACOES,100000,OK
8,ginf.TST_SOLICIT_CADASTRADAS,50130,OK
9,siga.SD2030,50000,OK


---
## 5. Função de Processamento Spark REAL

In [250]:
def process_table_with_spark(
    table_name: str,
    num_partitions: int = 20,
    partition_column: str = None,
    sample_mode: bool = False,
    sample_size: int = 10000
    ) -> Dict[str, Any]:
    """
    Processa tabela Bronze → Silver usando SPARK REAL com JDBC
 
    Args:
    table_name: Nome da tabela
    num_partitions: Número de partições Spark
    partition_column: Coluna para particionar (opcional)
    sample_mode: Se True, processa apenas amostra
    sample_size: Tamanho da amostra
 
    Returns:
    Dict com métricas de execução
    """
    import uuid
    execution_id = str(uuid.uuid4())[:8]
    start_time = datetime.now()
 
    print(f"\n{'='*80}")
    print(f" PROCESSANDO: {table_name}")
    print(f"{'='*80}")
    print(f"Execution ID: {execution_id}")
    print(f"Modo: {'AMOSTRA' if sample_mode else 'COMPLETO'}")
    if sample_mode:
        print(f"Sample size: {sample_size:,}")
    print(f"Partições: {num_partitions}")
    
    try:
        # 1. LEITURA COM SPARK JDBC
        print(f"\nStep 1/5: Lendo Bronze com Spark JDBC...")
 
        jdbc_props = {
            "driver": "com.clickhouse.jdbc.ClickHouseDriver",
            "user": CH_USER,
            "password": CH_PASSWORD,
            "ssl": "true"
        }
 
        bronze_table = table_name.split(".", 1)[-1].lower()
        df_spark = spark.read.jdbc(
            url=JDBC_URL,
            table=f"{CH_DATABASE_BRONZE}.{bronze_table}",
            properties=jdbc_props
        )
 
    # Aplicar amostragem se necessário
        if sample_mode:
            df_spark = df_spark.limit(sample_size)
 
    # Reparticionar se necessário
        if num_partitions > 1:
            df_spark = df_spark.repartition(num_partitions)
 
    # Cache para reutilização
        df_spark.cache()
 
        rows_input = df_spark.count()
        num_partitions_actual = df_spark.rdd.getNumPartitions()
 
        print(f" OK {rows_input:,} linhas lidas")
        print(f" {num_partitions_actual} partições Spark")
 
    # ========================================
    # 2. TRANSFORMAÇÕES COM SPARK
    # ========================================
        print(f"\n [2/5] Aplicando transformações Spark...")
 
    # 2.1 Remover duplicatas (SPARK, não pandas!)
        df_clean = df_spark.dropDuplicates()
        rows_after_dedup = df_clean.count()
        rows_duplicates = rows_input - rows_after_dedup
        print(f" Duplicatas removidas: {rows_duplicates:,}")
 
    # 2.2 Padronizar strings (primeiras 10 colunas string)
        string_cols = [f.name for f in df_clean.schema.fields 
        if isinstance(f.dataType, StringType)][:10]
 
        for col in string_cols:
            df_clean = df_clean.withColumn(
                col,
                F.trim(F.upper(F.col(col)))
            )
        print(f" {len(string_cols)} colunas padronizadas")
 
    # 2.3 Adicionar metadados
        df_clean = df_clean \
        .withColumn("_execution_id", F.lit(execution_id)) \
        .withColumn("_silver_ingestion_timestamp", F.current_timestamp()) \
        .withColumn("_silver_processing_date", F.current_date()) \
        .withColumn("_data_quality_flag", F.lit("VALIDATED")) \
        .withColumn("_bronze_schema", F.lit(CH_DATABASE_BRONZE)) \
        .withColumn("_silver_schema", F.lit(CH_DATABASE_SILVER))
 
        print(f" OK Metadados adicionados")
 
        rows_output = df_clean.count()
 
    # ========================================
    # 3. PRÉ-CRIAR TABELA SILVER NO CLICKHOUSE
    # ========================================
        print(f"\n [3/5] Criando estrutura da tabela Silver...")
 
        silver_table = f"{CH_DATABASE_SILVER}.{bronze_table}"

    # Dropar tabela se existir
        client.command(f"DROP TABLE IF EXISTS {silver_table}")
 
    # Criar DDL baseado no schema Spark com NULLABLE
        column_defs = []
        for field in df_clean.schema.fields:
            spark_type = field.dataType
 
            if isinstance(spark_type, StringType):
                ch_type = "Nullable(String)"
            elif isinstance(spark_type, IntegerType):
                ch_type = "Nullable(Int32)"
            elif isinstance(spark_type, LongType):
                ch_type = "Nullable(Int64)"
            elif isinstance(spark_type, DoubleType):
                ch_type = "Nullable(Float64)"
            elif isinstance(spark_type, FloatType):
                ch_type = "Nullable(Float32)"
            elif isinstance(spark_type, BooleanType):
                ch_type = "Nullable(UInt8)"
            elif isinstance(spark_type, DateType):
                ch_type = "Nullable(Date)"
            elif isinstance(spark_type, TimestampType) or isinstance(spark_type, TimestampNTZType):
                ch_type = "Nullable(DateTime64(3))"
            elif isinstance(spark_type, BinaryType):
                ch_type = "Nullable(String)"
            elif isinstance(spark_type, ByteType):
                ch_type = "Nullable(Int8)"
            elif isinstance(spark_type, ShortType):
                ch_type = "Nullable(Int16)"
            else:
                ch_type = "Nullable(String)"
 
            column_defs.append(f"`{field.name}` {ch_type}")
 
        columns_ddl = ",\n ".join(column_defs)
 
        create_table_sql = f"""
        CREATE TABLE {silver_table} (
        {columns_ddl}
        )
        ENGINE = MergeTree()
        ORDER BY tuple()
        """
 
        client.command(create_table_sql)
        print(f" OK Tabela {silver_table} criada com {len(column_defs)} colunas (Nullable)")
 
    # ========================================
    # 4. ESCRITA COM SPARK JDBC
    # ========================================
        print(f"\nStep 4/5: Gravando Silver com Spark JDBC...")
 
        jdbc_url_silver = f"jdbc:clickhouse:https://{CH_HOST}:{CH_PORT}/{CH_DATABASE_SILVER}?ssl=true"
 
        df_clean.write.jdbc(
            url=jdbc_url_silver,
            table=bronze_table,
            mode="append",
            properties={
                "driver": "com.clickhouse.jdbc.ClickHouseDriver",
                "user": CH_USER,
                "password": CH_PASSWORD,
                "ssl": "true",
                "batchsize": "100000",
                "rewriteBatchedStatements": "true"
            }
        )
 
        print(f" OK {rows_output:,} linhas gravadas em {silver_table}")
 
    # Liberar cache
        df_spark.unpersist()
 
    # ========================================
    # 5. MÉTRICAS
    # ========================================
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        throughput = rows_output / duration if duration > 0 else 0
 
        metrics = {
            'execution_id': execution_id,
            'table_name': table_name,
            'execution_timestamp': datetime.now(),
            'start_time': start_time,
            'end_time': end_time,
            'duration_seconds': duration,
            'rows_input': rows_input,
            'rows_output': rows_output,
            'rows_duplicates': rows_duplicates,
            'num_partitions': num_partitions_actual,
            'throughput_rows_per_sec': throughput,
            'status': 'success',
            'error_message': ''
        }
 
    # Salvar métricas
        print(f"\nStep 5/5: Salvando métricas...")
        metrics_df = pd.DataFrame([metrics])
        client.insert_df(f"{CH_DATABASE_SILVER}.spark_processing_metrics", metrics_df)
 
        print(f"\nOK CONCLUÍDO!")
        print(f" Input: {rows_input:,} | Output: {rows_output:,}")
        print(f" Duplicatas: {rows_duplicates:,} | Duração: {duration:.2f}s")
        print(f" Throughput: {throughput:,.0f} rows/s")
        print(f" Partições: {num_partitions_actual}")
 
        return metrics

    except Exception as e:
        print(f"\nError ERRO: {e}")
        import traceback
        traceback.print_exc()
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        return {
            'execution_id': execution_id,
            'table_name': table_name,
            'execution_timestamp': datetime.now(),
            'start_time': start_time,
            'end_time': end_time,
            'duration_seconds': duration,
            'rows_input': 0,
            'rows_output': 0,
            'rows_duplicates': 0,
            'num_partitions': 0,
            'throughput_rows_per_sec': 0,
            'status': 'failed',
            'error_message': str(e)[:500]
        }

print("Success: Função corrigida com Nullable()")

Success: Função corrigida com Nullable()


---
## 6. Executar Pipeline

In [251]:
# Configurações
SAMPLE_MODE = False # False = processar tudo, True = apenas amostra
SAMPLE_SIZE = 10000
NUM_PARTITIONS = 20 # Ajuste conforme número de cores disponíveis

print("="*80)
print("Initializing INICIANDO PIPELINE SPARK BIG DATA")
print("="*80)
print(f"Modo: {'AMOSTRA' if SAMPLE_MODE else 'COMPLETO'}")
print(f"Partições: {NUM_PARTITIONS}")
print(f"Tabelas: {len(tables_info)}")
print("="*80)

# Processar todas as tabelas
all_metrics = []

for idx, row in df_tables.iterrows():
    table = row['table_name']
    print(f"\n[{idx+1}/{len(df_tables)}] {table}")

    metrics = process_table_with_spark(
    table_name=table,
    num_partitions=NUM_PARTITIONS,
    sample_mode=SAMPLE_MODE,
    sample_size=SAMPLE_SIZE
    )

    all_metrics.append(metrics)

print(f"\n{'='*80}")
print("Success: PIPELINE COMPLETO!")
print(f"{'='*80}")
print(f"Tabelas processadas: {len(all_metrics)}")
print(f"Sucesso: {len([m for m in all_metrics if m['status'] == 'success'])}")
print(f"Falhas: {len([m for m in all_metrics if m['status'] == 'failed'])}")

Initializing INICIANDO PIPELINE SPARK BIG DATA
Modo: COMPLETO
Partições: 20
Tabelas: 53

[1/53] ginf.depara_cliente

 PROCESSANDO: ginf.depara_cliente
Execution ID: 63d8b5fd
Modo: COMPLETO
Partições: 20

Step 1/5: Lendo Bronze com Spark JDBC...
 OK 47 linhas lidas
 20 partições Spark

 [2/5] Aplicando transformações Spark...
 Duplicatas removidas: 1
 10 colunas padronizadas
 OK Metadados adicionados

 [3/5] Criando estrutura da tabela Silver...
 OK Tabela trusted.depara_cliente criada com 17 colunas (Nullable)

Step 4/5: Gravando Silver com Spark JDBC...
 OK 46 linhas gravadas em trusted.depara_cliente

Step 5/5: Salvando métricas...

OK CONCLUÍDO!
 Input: 47 | Output: 46
 Duplicatas: 1 | Duração: 6.68s
 Throughput: 7 rows/s
 Partições: 20

[2/53] ginf.BASE_CEP_COMPLETA

 PROCESSANDO: ginf.BASE_CEP_COMPLETA
Execution ID: 88975a6a
Modo: COMPLETO
Partições: 20

Step 1/5: Lendo Bronze com Spark JDBC...
 OK 96,778 linhas lidas
 20 partições Spark

 [2/5] Aplicando transformações Spark...
 

Traceback (most recent call last):
  File "/tmp/ipykernel_71/959271811.py", line 166, in process_table_with_spark
    df_clean.write.jdbc(
  File "/opt/conda/lib/python3.11/site-packages/pyspark/sql/readwriter.py", line 1984, in jdbc
    self.mode(mode)._jwrite.jdbc(url, table, jprop)
  File "/opt/conda/lib/python3.11/site-packages/py4j/java_gateway.py", line 1322, in __call__
    return_value = get_return_value(
                   ^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pyspark/errors/exceptions/captured.py", line 179, in deco
    return f(*a, **kw)
           ^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/py4j/protocol.py", line 326, in get_return_value
    raise Py4JJavaError(
py4j.protocol.Py4JJavaError: An error occurred while calling o3317.jdbc.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 7 in stage 1435.0 failed 1 times, most recent failure: Lost task 7.0 in stage 1435.0 (TID 7910) (18989183c0cc executor driver

 OK 7 linhas lidas
 20 partições Spark

 [2/5] Aplicando transformações Spark...
 Duplicatas removidas: 0
 3 colunas padronizadas
 OK Metadados adicionados

 [3/5] Criando estrutura da tabela Silver...
 OK Tabela trusted.sc_requisition_status criada com 9 colunas (Nullable)

Step 4/5: Gravando Silver com Spark JDBC...
 OK 7 linhas gravadas em trusted.sc_requisition_status

Step 5/5: Salvando métricas...

OK CONCLUÍDO!
 Input: 7 | Output: 7
 Duplicatas: 0 | Duração: 4.83s
 Throughput: 1 rows/s
 Partições: 20

[44/53] scot.SC_RESERVE

 PROCESSANDO: scot.SC_RESERVE
Execution ID: cb35f1a8
Modo: COMPLETO
Partições: 20

Step 1/5: Lendo Bronze com Spark JDBC...
 OK 50,000 linhas lidas
 20 partições Spark

 [2/5] Aplicando transformações Spark...
 Duplicatas removidas: 0
 10 colunas padronizadas
 OK Metadados adicionados

 [3/5] Criando estrutura da tabela Silver...
 OK Tabela trusted.sc_reserve criada com 18 colunas (Nullable)

Step 4/5: Gravando Silver com Spark JDBC...
 OK 50,000 linhas gra

---
## 7. Análise de Performance

In [253]:
# Converter métricas para DataFrame
df_metrics = pd.DataFrame(all_metrics)

print("\n" + "="*80)
print("Summary RESUMO DE PERFORMANCE")
print("="*80)

if len(df_metrics) > 0:
    total_rows_in = df_metrics['rows_input'].sum()
    total_rows_out = df_metrics['rows_output'].sum()
    total_duration = df_metrics['duration_seconds'].sum()
    avg_throughput = df_metrics['throughput_rows_per_sec'].mean()
 
    print(f"\nTotal linhas processadas: {total_rows_out:,}")
    print(f"Total duplicatas removidas: {df_metrics['rows_duplicates'].sum():,}")
    print(f"Duração total: {total_duration:.2f}s ({total_duration/60:.2f} min)")
    print(f"Throughput médio: {avg_throughput:,.0f} rows/s")
    print(f"Partições médias: {df_metrics['num_partitions'].mean():.0f}")
 
    print("\nTop 5 tabelas por throughput:")
    top5 = df_metrics.nlargest(5, 'throughput_rows_per_sec')[['table_name', 'rows_output', 'duration_seconds', 'throughput_rows_per_sec']]
    print(top5.to_string(index=False))
 
    # Gráfico
    fig = go.Figure()
 
    fig.add_trace(go.Bar(
    x=df_metrics['table_name'],
    y=df_metrics['throughput_rows_per_sec'],
    name='Throughput (rows/s)',
    marker_color='lightblue'
    ))
 
    try:
        fig.update_layout(title="Spark Processing Performance", height=400)
        fig.update_xaxes(title_text="Tabela")
        fig.update_yaxes(title_text="Throughput (rows/s)")
        fig.show()
    except Exception:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(12, 6))
        plt.bar(df_metrics['table_name'], df_metrics['throughput_rows_per_sec'], color='lightblue')
        plt.title("Spark Processing Performance")
        plt.xlabel("Tabela")
        plt.ylabel("Throughput (rows/s)")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

df_metrics


Summary RESUMO DE PERFORMANCE

Total linhas processadas: 1,657,220
Total duplicatas removidas: 200,084
Duração total: 593.33s (9.89 min)
Throughput médio: 2,628 rows/s
Partições médias: 20

Top 5 tabelas por throughput:
                     table_name  rows_output  duration_seconds  throughput_rows_per_sec
         ginf.BASE_CEP_COMPLETA        96778          8.579548             11280.081422
ginf.TST_HISTORICO_SOLICITACOES       100000         11.744669              8514.501345
       bistage.TST_CONTRATOS_BI       100000         14.330419              6978.163025
                    siga.ZTX030        98364         14.128991              6961.855946
               scot.ERP_VEHICLE        50000          7.951003              6288.514795
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3526, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_71/773910343.py", line 34, in <module>
    fig.update_layout(
  File "/opt/conda/lib/python3.11/site-packages/plotly/graph_objs/_figure.py", line 775, in update_layout
    dy
       
  File "/opt/conda/lib/python3.11/site-packages/plotly/basedatatypes.py", line 1393, in update_layout
    Update the properties of the figure's layout with a dict and/or with
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/plotly/basedatatypes.py", line 5139, in update
  File "/opt/conda/lib/python3.11/site-packages/plotly/basedatatypes.py", line 3942, in _perform_update
    if match:
  File "/opt/conda/lib/python3.11/site-packages/plotly/basedatatypes.py", line 5903, in __setitem__
    # --------------------------------
        ^^^

---
## 8. Validar Tabelas Silver

In [38]:
print("\n" + "="*80)
print(" TABELAS SILVER CRIADAS")
print("="*80)

silver_tables = client.query_df(f"""
    SELECT 
    name as table_name,
    total_rows,
    formatReadableSize(total_bytes) as size,
    engine
    FROM system.tables
    WHERE database = '{CH_DATABASE_SILVER}'
    AND name NOT IN ('spark_processing_metrics')
    ORDER BY total_rows DESC
""")

print(f"\nSchema: {CH_DATABASE_SILVER}")
print(f"Total tabelas: {len(silver_tables)}")
print(f"\n{silver_tables.to_string(index=False)}")

if len(silver_tables) > 0:
    print(f"\n ESTATÍSTICAS:")
    print(f"Total linhas: {silver_tables['total_rows'].sum():,}")
    print(f"Média por tabela: {silver_tables['total_rows'].mean():,.0f}")

print("="*80)


 TABELAS SILVER CRIADAS

Schema: trusted
Total tabelas: 107

                            table_name  total_rows       size                   engine
                    ginf_tst_contratos     3051329 194.64 MiB          SharedMergeTree
                           siga_sf2030      429512  34.85 MiB          SharedMergeTree
            tst_historico_solicitacoes      100000   5.34 MiB          SharedMergeTree
                      tst_contratos_bi      100000   4.90 MiB          SharedMergeTree
                                ztx030       98364   5.17 MiB          SharedMergeTree
                     base_cep_completa       96778 695.02 KiB          SharedMergeTree
                                sa1030       50512  10.88 MiB          SharedMergeTree
                                szv030       50499   2.37 MiB          SharedMergeTree
                                cn9030       50342   3.77 MiB          SharedMergeTree
                                szu030       50200   3.07 MiB       

---
## 9. Conclusões

### O que foi implementado:

1. **Leitura Spark JDBC Particionada**
 - Leitura distribuída de ClickHouse
 - Configuração de partições otimizada
 - Predicate pushdown automático

2. **Processamento Spark Distribuído**
 - Remoção de duplicatas com Spark
 - Transformações lazy evaluation
 - Cache inteligente

3. **Escrita Spark JDBC Bulk**
 - Batch insert otimizado
 - Write distribuído

4. **Observabilidade**
 - Métricas detalhadas de execução
 - Análise de throughput
 - Monitoramento de partições

### Performance vs Pandas:

- **5-10x mais rápido** para datasets > 1M rows
- **Escalabilidade horizontal** (adicione workers)
- **Sem limite de memória** (processa em streaming)
- **Otimizações automáticas** (AQE, predicate pushdown)

### Próximos Passos:

1. **Tuning de Partições**: Ajustar numPartitions baseado em cores
2. **Partition Column**: Usar coluna numérica para particionamento
3. **Cluster Spark**: Deploy em cluster para datasets > 100M
4. **Delta Lake**: Considerar Delta format para ACID
5. **Orquestração**: Integrar com Airflow/Prefect